# 1. Liver masks and crops

Run on Google Colab with a GPU.

**Needs**
- the CARE liver training archive `Liver-Train.zip` in `MyDrive`
- the trained segmentation model in `MyDrive/CARE2026_ckpts/liseg_v1`, made by `segmentation/liver_segmentation.ipynb`

**Makes** (in `MyDrive/CARE2026_ckpts`)
- `liseg_v1/lifs_masks`: liver masks for all 460 examinations from the final segmentation weights (`checkpoint_final.pth`), used for the radiomic features
- `lifs_roi_cache`: crops of all seven series around those masks
- `lifs_train_masks_extra` and `lifs_train_masks_st2_345`: masks from the selected checkpoint (`checkpoint_best.pth`), used for the deep crops
- `lifs_ged4_hires_train_v2`: the GED4 crops the deep stream is trained on

`lifs_ged4_hires_train` is an earlier crop version. Later steps only use it to list the cases.

**The extraction error is in this notebook on purpose.** Step 9 is the crop code described in Sec. 4.2 of the paper: it anchors the crop grid at a corner chosen in physical coordinates, and for some image orientations the grid misses the liver. In 115 of the 460 examinations the cropped mask comes out empty. We kept the code as it ran, because Table 2 shows what this error does. Notebook 2 (Step 6) rebuilds the radiomic features with the fix.

**Step 1.** Mount Google Drive and set the project folder.

In [ ]:
# Settings: mount Google Drive and set the project folder
from google.colab import drive
drive.mount('/content/drive')
CKPT = "/content/drive/MyDrive/CARE2026_ckpts"

**Step 2.** Copy `Liver-Train.zip` from Drive to the Colab disk.

In [ ]:
import shutil, os
shutil.copy("/content/drive/MyDrive/Liver-Train.zip", "/content/Liver-Train.zip")
print("copied", os.path.getsize("/content/Liver-Train.zip"))

**Step 3.** Extract it.

In [ ]:
import zipfile
with zipfile.ZipFile("/content/Liver-Train.zip") as z:
    z.extractall("/content/Liver-Train-data")
print("done")

**Step 4.** Install nnU-Net v2.

In [ ]:
!pip install -q nnunetv2 2>&1 | tail -3

**Step 5.** nnU-Net needs numpy below 2.1.

In [ ]:
!pip install -q "numpy<2.1" 2>&1 | tail -2

**Step 6.** Register the custom nnU-Net trainer classes, so the segmentation checkpoints can be loaded.

In [ ]:
import os, nnunetv2
nn_dir = os.path.dirname(nnunetv2.__file__)
tdir = f"{nn_dir}/training/nnUNetTrainer/variants/training_length"
os.makedirs(tdir, exist_ok=True)

with open(f"{tdir}/nnUNetTrainer_250epochs.py", "w") as f:
    f.write(
"from nnunetv2.training.nnUNetTrainer.nnUNetTrainer import nnUNetTrainer\n"
"import torch\n"
"class nnUNetTrainer_250epochs(nnUNetTrainer):\n"
"    def __init__(self, plans, configuration, fold, dataset_json, device=torch.device('cuda')):\n"
"        super().__init__(plans, configuration, fold, dataset_json, device)\n"
"        self.num_epochs = 250\n")

for s in [123, 7]:
    with open(f"{tdir}/nnUNetTrainer_250epochs_seed{s}.py", "w") as f:
        f.write(
f"from nnunetv2.training.nnUNetTrainer.nnUNetTrainer import nnUNetTrainer\n"
f"import torch, numpy as np, random\n"
f"class nnUNetTrainer_250epochs_seed{s}(nnUNetTrainer):\n"
f"    def __init__(self, plans, configuration, fold, dataset_json, device=torch.device('cuda')):\n"
f"        random.seed({s}); np.random.seed({s}); torch.manual_seed({s}); torch.cuda.manual_seed_all({s})\n"
f"        super().__init__(plans, configuration, fold, dataset_json, device)\n"
f"        self.num_epochs = 250\n")

with open(f"{tdir}/nnUNetTrainerST2.py", "w") as f:
    f.write(
"from nnunetv2.training.nnUNetTrainer.nnUNetTrainer import nnUNetTrainer\n"
"import torch\n"
"class nnUNetTrainerST2(nnUNetTrainer):\n"
"    def __init__(self, plans, configuration, fold, dataset_json, device=torch.device('cuda')):\n"
"        super().__init__(plans, configuration, fold, dataset_json, device)\n")

src = "/content/drive/MyDrive/CARE2026_ckpts/liseg_v1/code/nnUNetTrainerStage3.py"
import shutil
shutil.copy(src, f"{tdir}/nnUNetTrainerStage3.py")

print("registered:", sorted(os.listdir(tdir)))

**Step 7.** Point nnU-Net to the trained segmentation model.

In [ ]:
import os
base = "/content/drive/MyDrive/CARE2026_ckpts/liseg_v1"
os.environ["nnUNet_raw"] = "/content/nnUNet_raw"
os.environ["nnUNet_preprocessed"] = "/content/nnUNet_preprocessed"
os.environ["nnUNet_results"] = f"{base}/nnUNet_results"
os.makedirs("/content/nnUNet_raw", exist_ok=True)
os.makedirs("/content/nnUNet_preprocessed", exist_ok=True)
print("env set")

**Step 8.** Liver masks for all 460 examinations from the final weights (`liseg_v1/lifs_masks`). GPU, resumable.

In [ ]:
import os, time, shutil, torch, pandas as pd
from nnunetv2.inference.predict_from_raw_data import nnUNetPredictor

DATA = "/content/Liver-Train-data/Liver-Train"
MASK_DIR = "/content/drive/MyDrive/CARE2026_ckpts/liseg_v1/lifs_masks"
STAGE_IN = "/content/seg_stage_in"
STAGE_OUT = "/content/seg_stage_out"
os.makedirs(MASK_DIR, exist_ok=True)

df = pd.read_csv(f"{DATA}/info_2026.csv")
todo = []
for _, r in df.iterrows():
    fn = r['filename']
    if os.path.exists(f"{MASK_DIR}/{fn}.nii.gz"):
        continue
    todo.append((fn, r['vendor']))

print(f"total {len(df)}, already done {len(df)-len(todo)}, to predict {len(todo)}")

if len(todo) > 0:
    predictor = nnUNetPredictor(tile_step_size=0.5, use_gaussian=True, use_mirroring=True,
                                device=torch.device('cuda'), verbose=False, allow_tqdm=False)
    mdir = f"{os.environ['nnUNet_results']}/Dataset503_LiverST2/nnUNetTrainerST2__nnUNetResEncUNetLPlans__3d_fullres"
    predictor.initialize_from_trained_model_folder(mdir, use_folds=(0,), checkpoint_name="checkpoint_final.pth")
    print("ST2 loaded, starting")

    t_start = time.time()
    for i, (fn, vendor) in enumerate(todo):
        shutil.rmtree(STAGE_IN, ignore_errors=True)
        shutil.rmtree(STAGE_OUT, ignore_errors=True)
        os.makedirs(STAGE_IN, exist_ok=True)
        os.makedirs(STAGE_OUT, exist_ok=True)
        src = f"{DATA}/Vendor_{vendor}/{fn}/GED4.nii.gz"
        shutil.copy(src, f"{STAGE_IN}/{fn}_0000.nii.gz")

        predictor.predict_from_files(STAGE_IN, STAGE_OUT, save_probabilities=False,
                                     overwrite=True, num_processes_preprocessing=1,
                                     num_processes_segmentation_export=1)

        produced = f"{STAGE_OUT}/{fn}.nii.gz"
        tmp = f"{MASK_DIR}/{fn}.nii.gz.tmp"
        final = f"{MASK_DIR}/{fn}.nii.gz"
        shutil.copy(produced, tmp)
        os.replace(tmp, final)

        done = i + 1
        elapsed = time.time() - t_start
        rate = elapsed / done
        eta = rate * (len(todo) - done) / 60.0
        print(f"[{done}/{len(todo)}] {fn}  ({rate:.0f}s/case, ETA {eta:.0f} min)", flush=True)

print("done with this batch")
n_have = len([f for f in os.listdir(MASK_DIR) if f.endswith('.nii.gz')])
print(f"masks on drive: {n_have}/{len(df)}")

**Step 9.** Crops of all seven series around each mask (`lifs_roi_cache`). This is the code with the extraction error of Sec. 4.2 (see the note at the top): 115 crops come out empty. Notebook 2 fixes it in Step 6.

In [ ]:
import os, time, json, shutil, numpy as np, pandas as pd, SimpleITK as sitk

DATA = "/content/Liver-Train-data/Liver-Train"
MASK_DIR = "/content/drive/MyDrive/CARE2026_ckpts/liseg_v1/lifs_masks"
ROI_DIR = "/content/drive/MyDrive/CARE2026_ckpts/lifs_roi_cache"
os.makedirs(ROI_DIR, exist_ok=True)

PHASES = ['T1', 'T2', 'DWI_800', 'GED1', 'GED2', 'GED3', 'GED4']
OUT_SHAPE = (64, 128, 128)
MARGIN = 0.10

def physical_bbox_from_mask(mask_img, margin):
    arr = sitk.GetArrayFromImage(mask_img) > 0
    zz, yy, xx = np.where(arr)
    if len(zz) == 0:
        return None
    lo_idx = [int(xx.min()), int(yy.min()), int(zz.min())]
    hi_idx = [int(xx.max()), int(yy.max()), int(zz.max())]
    span = [hi_idx[d] - lo_idx[d] + 1 for d in range(3)]
    pad = [int(round(span[d] * margin)) for d in range(3)]
    size_full = mask_img.GetSize()
    lo_idx = [max(0, lo_idx[d] - pad[d]) for d in range(3)]
    hi_idx = [min(size_full[d] - 1, hi_idx[d] + pad[d]) for d in range(3)]
    p_lo = mask_img.TransformIndexToPhysicalPoint(lo_idx)
    p_hi = mask_img.TransformIndexToPhysicalPoint(hi_idx)
    origin = [min(p_lo[d], p_hi[d]) for d in range(3)]
    far = [max(p_lo[d], p_hi[d]) for d in range(3)]
    phys_size = [far[d] - origin[d] for d in range(3)]
    return origin, phys_size, mask_img.GetDirection()

def resample_to_grid(img, origin, phys_size, direction, out_shape, is_mask):
    out_size = [out_shape[2], out_shape[1], out_shape[0]]
    out_spacing = [phys_size[d] / max(out_size[d], 1) for d in range(3)]
    out_spacing = [s if s > 1e-6 else 1.0 for s in out_spacing]
    rs = sitk.ResampleImageFilter()
    rs.SetOutputOrigin(origin)
    rs.SetOutputDirection(direction)
    rs.SetOutputSpacing(out_spacing)
    rs.SetSize([int(v) for v in out_size])
    rs.SetInterpolator(sitk.sitkNearestNeighbor if is_mask else sitk.sitkLinear)
    rs.SetDefaultPixelValue(0)
    return rs.Execute(img)

def normalize(vol, mask):
    m = mask > 0
    if m.sum() > 0:
        mu = vol[m].mean(); sd = vol[m].std() + 1e-6
    else:
        nz = vol > 0
        if nz.sum() == 0:
            return vol.astype(np.float32)
        mu = vol[nz].mean(); sd = vol[nz].std() + 1e-6
    return ((vol - mu) / sd).astype(np.float32)

df = pd.read_csv(f"{DATA}/info_2026.csv")
todo = [r for _, r in df.iterrows()
        if not os.path.exists(f"{ROI_DIR}/{r['filename']}.npz")]
print(f"total {len(df)}, done {len(df)-len(todo)}, to process {len(todo)}")

t0 = time.time()
for i, r in enumerate(todo):
    fn, vendor = r['filename'], r['vendor']
    mask_path = f"{MASK_DIR}/{fn}.nii.gz"
    try:
        mask_img = sitk.ReadImage(mask_path, sitk.sitkUInt8)
        bbox = physical_bbox_from_mask(mask_img, MARGIN)
        if bbox is None:
            print(f"  SKIP {fn}: empty mask", flush=True)
            continue
        origin, phys_size, direction = bbox

        rs_mask = resample_to_grid(mask_img, origin, phys_size, direction, OUT_SHAPE, True)
        mask_arr = (sitk.GetArrayFromImage(rs_mask) > 0).astype(np.uint8)

        vol = np.zeros((len(PHASES),) + OUT_SHAPE, dtype=np.float32)
        avail = np.zeros(len(PHASES), dtype=np.float32)
        for ci, ph in enumerate(PHASES):
            p = f"{DATA}/Vendor_{vendor}/{fn}/{ph}.nii.gz"
            if not os.path.exists(p):
                continue
            try:
                img = sitk.ReadImage(p, sitk.sitkFloat32)
                rs = resample_to_grid(img, origin, phys_size, direction, OUT_SHAPE, False)
                a = sitk.GetArrayFromImage(rs).astype(np.float32)
                vol[ci] = normalize(a, mask_arr)
                avail[ci] = 1.0
            except Exception as e:
                avail[ci] = 0.0

        local_tmp = "/content/_roi_tmp.npz"
        final = f"{ROI_DIR}/{fn}.npz"
        np.savez_compressed(local_tmp, vol=vol, mask=mask_arr, avail=avail,
                            stage=int(str(fn).split('-S')[-1]),
                            vendor=str(vendor), filename=str(fn))
        shutil.copy(local_tmp, final)

        done = i + 1
        el = time.time() - t0
        eta = el / done * (len(todo) - done) / 60.0
        print(f"[{done}/{len(todo)}] {fn} avail={int(avail.sum())}/7 "
              f"({el/done:.1f}s/case, ETA {eta:.0f} min)", flush=True)
    except Exception as e:
        print(f"  ERROR {fn}: {e}", flush=True)

n_have = len([f for f in os.listdir(ROI_DIR) if f.endswith('.npz')])
print(f"ROI cache on drive: {n_have}/{len(df)}")

**Step 10.** Extract GED4 and the case table.

In [ ]:
# === Cell 1b: selective extract — GED4 + info CSV only ===
import zipfile, os, glob, time

zp  = "/content/drive/MyDrive/Liver-Train.zip"
dst = "/content/Liver-Train"
os.makedirs(dst, exist_ok=True)

t0 = time.time()
with zipfile.ZipFile(zp) as z:
    members = [n for n in z.namelist()
               if n.endswith("GED4.nii.gz") or n.endswith("info_2026.csv")]
    print(f"extracting {len(members)} files (GED4 + csv only, not full 33GB)...")
    for i, m in enumerate(members, 1):
        z.extract(m, dst)
        if i % 50 == 0:
            print(f"  {i}/{len(members)}  ({time.time()-t0:.0f}s)")
print(f"✓ extracted {len(members)} files in {time.time()-t0:.0f}s")

# verify
g = glob.glob(f"{dst}/Liver-Train/**/GED4.nii.gz", recursive=True)
print(f"\nGED4 files on disk: {len(g)}")
print("sample:", g[0] if g else "NONE")

# check if training masks already exist on Drive
print("\n=== training liver masks — already cached? ===")
mask_candidates = [
    f"{CKPT}/lifs_roi_cache",          # ROI cache (may contain masks)
    f"{CKPT}/liseg_v1/train_masks",
    f"{CKPT}/lifs_train_masks",
]
for mc in mask_candidates:
    n = len(glob.glob(f"{mc}/*")) if os.path.exists(mc) else 0
    print(f"  {'✓' if n else '✗'} {mc}  ({n} items)")

**Step 11.** First GED4 crops, from the cached masks (`lifs_ged4_hires_train`).

In [ ]:
# === Cell 2: high-res GED4 crop (Option A — upsampled cached mask as bbox) ===
import numpy as np, glob, os, SimpleITK as sitk, time

ROI_CACHE = f"{CKPT}/lifs_roi_cache"
RAW_ROOT  = "/content/Liver-Train/Liver-Train"
OUT_DIR   = f"{CKPT}/lifs_ged4_hires_train"
os.makedirs(OUT_DIR, exist_ok=True)

HI = (24, 256, 256)   # (Z, Y, X) high-res target
PAD_FRAC = 0.10        # 10% margin around liver bbox (keeps capsule/surface)

def raw_ged4_path(fname):
    # fname like '0888-A-S1'; vendor letter after first dash segment
    hits = glob.glob(f"{RAW_ROOT}/Vendor_*/{fname}/GED4.nii.gz")
    return hits[0] if hits else None

roi_files = sorted(glob.glob(f"{ROI_CACHE}/*.npz"))
print(f"processing {len(roi_files)} cases...")
t0 = time.time(); done = 0; skipped = 0

for rp in roi_files:
    fname = os.path.basename(rp).replace(".npz","")
    out_p = f"{OUT_DIR}/{fname}.npz"
    if os.path.exists(out_p):
        skipped += 1; done += 1; continue

    gp = raw_ged4_path(fname)
    if gp is None:
        print(f"  ✗ no raw GED4: {fname}"); continue

    cache = np.load(rp)
    lowmask = cache["mask"]            # (64,128,128) uint8
    stage   = int(cache["stage"])
    vendor  = str(cache["vendor"])
    avail   = cache["avail"]

    img = sitk.ReadImage(gp, sitk.sitkFloat32)
    raw = sitk.GetArrayFromImage(img)  # (Z,Y,X) native

    # upsample low-res mask to raw shape (nearest), get bbox
    lm = sitk.GetImageFromArray(lowmask)
    lm_r = sitk.Resample(lm, [raw.shape[2], raw.shape[1], raw.shape[0]],
                         sitk.Transform(), sitk.sitkNearestNeighbor,
                         lm.GetOrigin(),
                         [lowmask.shape[2]/raw.shape[2],
                          lowmask.shape[1]/raw.shape[1],
                          lowmask.shape[0]/raw.shape[0]],
                         lm.GetDirection(), 0, sitk.sitkUInt8)
    mr = sitk.GetArrayFromImage(lm_r) > 0
    if mr.sum() == 0:
        print(f"  ✗ empty mask: {fname}"); continue

    zz, yy, xx = np.where(mr)
    def span(a, lo, hi, n):
        pad = int((hi - lo) * PAD_FRAC)
        return max(0, lo - pad), min(n, hi + pad + 1)
    z0, z1 = span(zz, zz.min(), zz.max(), raw.shape[0])
    y0, y1 = span(yy, yy.min(), yy.max(), raw.shape[1])
    x0, x1 = span(xx, xx.min(), xx.max(), raw.shape[2])

    crop = raw[z0:z1, y0:y1, x0:x1]
    ci = sitk.GetImageFromArray(crop)
    ci_r = sitk.Resample(ci, [HI[2], HI[1], HI[0]],
                         sitk.Transform(), sitk.sitkLinear,
                         ci.GetOrigin(),
                         [crop.shape[2]/HI[2], crop.shape[1]/HI[1], crop.shape[0]/HI[0]],
                         ci.GetDirection(), 0, sitk.sitkFloat32)
    out = sitk.GetArrayFromImage(ci_r)
    out = (out - out.mean()) / (out.std() + 1e-6)

    np.savez_compressed(out_p,
                        ged4=out.astype(np.float32),
                        stage=stage, vendor=vendor, avail=avail)
    done += 1
    if done % 25 == 0:
        print(f"  {done}/{len(roi_files)}  ({time.time()-t0:.0f}s)")

print(f"\n✓ done: {done}/{len(roi_files)} (skipped {skipped} cached) in {time.time()-t0:.0f}s")
print(f"output: {OUT_DIR}")
# sanity
s = sorted(glob.glob(f"{OUT_DIR}/*.npz"))
if s:
    z = np.load(s[0])
    print("sample keys:", z.files, "| ged4 shape:", z["ged4"].shape, "| stage:", int(z["stage"]))

**Step 12.** nnU-Net setup for the selected checkpoint.

In [ ]:
# === Cell 3a: nnU-Net env for LiSeg ST2 inference ===
!pip install -q nnunetv2 2>&1 | tail -2
!pip install -q "numpy<2.1" 2>&1 | tail -1

import os, nnunetv2, torch
base = f"{CKPT}/liseg_v1"
os.environ["nnUNet_raw"]          = "/content/nnUNet_raw"
os.environ["nnUNet_preprocessed"] = "/content/nnUNet_preprocessed"
os.environ["nnUNet_results"]      = f"{base}/nnUNet_results"
os.makedirs("/content/nnUNet_raw", exist_ok=True)
os.makedirs("/content/nnUNet_preprocessed", exist_ok=True)

# register ST2 trainer stub so the checkpoint loads
nn_dir = os.path.dirname(nnunetv2.__file__)
tdir = f"{nn_dir}/training/nnUNetTrainer/variants/training_length"
os.makedirs(tdir, exist_ok=True)
with open(f"{tdir}/nnUNetTrainerST2.py","w") as f:
    f.write("from nnunetv2.training.nnUNetTrainer.nnUNetTrainer import nnUNetTrainer\n"
            "import torch\n"
            "class nnUNetTrainerST2(nnUNetTrainer):\n"
            "    def __init__(self, plans, configuration, fold, dataset_json, device=torch.device('cuda')):\n"
            "        super().__init__(plans, configuration, fold, dataset_json, device)\n")

# confirm the ST2 model folder exists
import glob
mdl = glob.glob(f"{base}/nnUNet_results/**/nnUNetTrainerST2*/", recursive=True)
print("env set | cuda:", torch.cuda.is_available())
print("ST2 model folder:", mdl[0] if mdl else "✗ NOT FOUND — need to check path")

**Step 13.** Masks from the selected checkpoint for the 115 examinations whose cached mask was empty (`lifs_train_masks_extra`).

In [ ]:
# === Cell 3b: pseudo-mask the 115 missing cases with ST2 ===
import numpy as np, glob, os, shutil, time

# the 115 empty-mask cases (recompute to be safe)
cache_files = sorted(glob.glob(f"{CKPT}/lifs_roi_cache/*.npz"))
missing = []
for cp in cache_files:
    fname = os.path.basename(cp).replace(".npz","")
    if int((np.load(cp)["mask"] > 0).sum()) == 0:
        missing.append(fname)
print(f"cases needing masks: {len(missing)}")

RAW_ROOT = "/content/Liver-Train/Liver-Train"
IN_DIR   = "/content/st2_infer_in"
OUT_DIR  = "/content/st2_infer_out"
MASK_SAVE = f"{CKPT}/lifs_train_masks_extra"   # where new masks live
for d in [IN_DIR, OUT_DIR, MASK_SAVE]:
    os.makedirs(d, exist_ok=True)

# stage GED4 files in nnU-Net naming: {case}_0000.nii.gz
def raw_ged4(fname):
    h = glob.glob(f"{RAW_ROOT}/Vendor_*/{fname}/GED4.nii.gz")
    return h[0] if h else None

staged = 0
for fname in missing:
    gp = raw_ged4(fname)
    if gp is None:
        print(f"  ✗ no raw GED4: {fname}"); continue
    dst = f"{IN_DIR}/{fname}_0000.nii.gz"
    if not os.path.exists(dst):
        shutil.copy(gp, dst)
    staged += 1
print(f"staged {staged} GED4 files for inference")

# run ST2 inference
from nnunetv2.inference.predict_from_raw_data import nnUNetPredictor
import torch

t0 = time.time()
predictor = nnUNetPredictor(
    tile_step_size=0.5, use_gaussian=True, use_mirroring=True,
    device=torch.device('cuda'), verbose=False, allow_tqdm=True)
predictor.initialize_from_trained_model_folder(
    f"{CKPT}/liseg_v1/nnUNet_results/Dataset503_LiverST2/nnUNetTrainerST2__nnUNetResEncUNetLPlans__3d_fullres",
    use_folds=(0,), checkpoint_name="checkpoint_best.pth")
print("predictor ready, running inference...")

predictor.predict_from_files(
    IN_DIR, OUT_DIR,
    save_probabilities=False, overwrite=False,
    num_processes_preprocessing=2, num_processes_segmentation_export=2)

# copy masks to Drive
import glob as g
made = g.glob(f"{OUT_DIR}/*.nii.gz")
for m in made:
    shutil.copy(m, f"{MASK_SAVE}/{os.path.basename(m)}")
print(f"\n✓ inference done in {time.time()-t0:.0f}s | masks: {len(made)} → {MASK_SAVE}")

# quick check: are the new masks non-empty?
import SimpleITK as sitk
nonempty = 0
for m in made[:10]:
    arr = sitk.GetArrayFromImage(sitk.ReadImage(m))
    if (arr > 0).sum() > 100: nonempty += 1
print(f"sanity (first 10): {nonempty}/10 have non-trivial liver masks")

**Step 14.** First GED4 crops for those 115 examinations.

In [ ]:
# === Cell 3c: high-res GED4 crop for the 115 newly-masked cases ===
import numpy as np, glob, os, SimpleITK as sitk, time

MASK_EXTRA = f"{CKPT}/lifs_train_masks_extra"
RAW_ROOT   = "/content/Liver-Train/Liver-Train"
OUT_DIR    = f"{CKPT}/lifs_ged4_hires_train"   # SAME output dir — they join the 345
ROI_CACHE  = f"{CKPT}/lifs_roi_cache"          # for stage/vendor/avail labels

HI = (24, 256, 256)
PAD_FRAC = 0.10

def raw_ged4(fname):
    h = glob.glob(f"{RAW_ROOT}/Vendor_*/{fname}/GED4.nii.gz")
    return h[0] if h else None

mask_files = sorted(glob.glob(f"{MASK_EXTRA}/*.nii.gz"))
print(f"re-cropping {len(mask_files)} newly-masked cases...")
t0 = time.time(); done = 0

for mp in mask_files:
    fname = os.path.basename(mp).replace(".nii.gz","")
    out_p = f"{OUT_DIR}/{fname}.npz"
    if os.path.exists(out_p):
        done += 1; continue

    gp = raw_ged4(fname)
    if gp is None:
        print(f"  ✗ no raw GED4: {fname}"); continue

    # labels from the ROI cache (stage/vendor/avail still valid there)
    cache = np.load(f"{ROI_CACHE}/{fname}.npz")
    stage  = int(cache["stage"]); vendor = str(cache["vendor"]); avail = cache["avail"]

    img  = sitk.ReadImage(gp, sitk.sitkFloat32)
    mask = sitk.ReadImage(mp)
    # resample mask onto GED4 grid (ST2 mask is already in GED4 space, but ensure match)
    mask = sitk.Resample(mask, img, sitk.Transform(), sitk.sitkNearestNeighbor, 0)
    raw  = sitk.GetArrayFromImage(img)
    mr   = sitk.GetArrayFromImage(mask) > 0
    if mr.sum() == 0:
        print(f"  ✗ still empty after resample: {fname}"); continue

    zz, yy, xx = np.where(mr)
    def span(lo, hi, n):
        pad = int((hi - lo) * PAD_FRAC)
        return max(0, lo - pad), min(n, hi + pad + 1)
    z0,z1 = span(zz.min(), zz.max(), raw.shape[0])
    y0,y1 = span(yy.min(), yy.max(), raw.shape[1])
    x0,x1 = span(xx.min(), xx.max(), raw.shape[2])

    crop = raw[z0:z1, y0:y1, x0:x1]
    ci = sitk.GetImageFromArray(crop)
    ci_r = sitk.Resample(ci, [HI[2],HI[1],HI[0]], sitk.Transform(), sitk.sitkLinear,
                         ci.GetOrigin(),
                         [crop.shape[2]/HI[2], crop.shape[1]/HI[1], crop.shape[0]/HI[0]],
                         ci.GetDirection(), 0, sitk.sitkFloat32)
    out = sitk.GetArrayFromImage(ci_r)
    out = (out - out.mean()) / (out.std() + 1e-6)

    np.savez_compressed(out_p, ged4=out.astype(np.float32),
                        stage=stage, vendor=vendor, avail=avail)
    done += 1
    if done % 25 == 0: print(f"  {done}/{len(mask_files)}  ({time.time()-t0:.0f}s)")

print(f"\n✓ re-cropped {done}/{len(mask_files)} in {time.time()-t0:.0f}s")

# === FULL SET VERIFICATION ===
all_files = sorted(glob.glob(f"{OUT_DIR}/*.npz"))
print(f"\n=== high-res GED4 training set now: {len(all_files)} cases ===")
from collections import Counter
stages, vendors = [], []
for f in all_files:
    d = np.load(f)
    stages.append(int(d["stage"])); vendors.append(str(d["vendor"]))
print("by stage:", dict(sorted(Counter(stages).items())))
print("by vendor:", dict(Counter(vendors)))
print("target: 460 total")

**Step 15.** Extract the GED4 files again.

In [ ]:
# === Cell A: re-extract GED4 files from Liver-Train.zip ===
import zipfile, os, glob, time
zp="/content/drive/MyDrive/Liver-Train.zip"; dst="/content/Liver-Train"
os.makedirs(dst,exist_ok=True)
t0=time.time()
with zipfile.ZipFile(zp) as z:
    members=[n for n in z.namelist() if n.endswith("GED4.nii.gz")]
    print(f"extracting {len(members)} GED4 files...")
    for i,m in enumerate(members,1):
        z.extract(m,dst)
        if i%100==0: print(f"  {i}/{len(members)} ({time.time()-t0:.0f}s)")
print(f"✓ {len(members)} files in {time.time()-t0:.0f}s")
g=glob.glob(f"{dst}/**/GED4.nii.gz",recursive=True); print(f"GED4 on disk: {len(g)}")

**Step 16.** Masks from the selected checkpoint for the other 345 examinations (`lifs_train_masks_st2_345`).

In [ ]:
# === Cell B: ST2 masks for the 345 cases (nnUNet env + inference) ===
!pip install -q nnunetv2 "numpy<2.1" 2>&1 | tail -1
import os, glob, shutil, numpy as np, nnunetv2, torch

base=f"{CKPT}/liseg_v1"
os.environ["nnUNet_raw"]="/content/nnUNet_raw"; os.environ["nnUNet_preprocessed"]="/content/nnUNet_preprocessed"
os.environ["nnUNet_results"]=f"{base}/nnUNet_results"
for d in ["/content/nnUNet_raw","/content/nnUNet_preprocessed"]: os.makedirs(d,exist_ok=True)
nn_dir=os.path.dirname(nnunetv2.__file__)
tdir=f"{nn_dir}/training/nnUNetTrainer/variants/training_length"; os.makedirs(tdir,exist_ok=True)
with open(f"{tdir}/nnUNetTrainerST2.py","w") as f:
    f.write("from nnunetv2.training.nnUNetTrainer.nnUNetTrainer import nnUNetTrainer\nimport torch\n"
            "class nnUNetTrainerST2(nnUNetTrainer):\n"
            "    def __init__(self,plans,configuration,fold,dataset_json,device=torch.device('cuda')):\n"
            "        super().__init__(plans,configuration,fold,dataset_json,device)\n")

# which cases need masks
extra=set(os.path.basename(f).replace(".nii.gz","") for f in glob.glob(f"{CKPT}/lifs_train_masks_extra/*.nii.gz"))
allc=set(os.path.basename(f).replace(".npz","") for f in glob.glob(f"{CKPT}/lifs_ged4_hires_train/*.npz"))
need=sorted(allc-extra)
print(f"need ST2 masks for {len(need)} cases")

# stage GED4 in nnUNet naming
IN="/content/st2_345_in"; OUT="/content/st2_345_out"; SAVE=f"{CKPT}/lifs_train_masks_st2_345"
for d in [IN,OUT,SAVE]: os.makedirs(d,exist_ok=True)
RAW="/content/Liver-Train"
for case in need:
    h=glob.glob(f"{RAW}/**/{case}/GED4.nii.gz",recursive=True)
    if h:
        dst=f"{IN}/{case}_0000.nii.gz"
        if not os.path.exists(dst): shutil.copy(h[0],dst)
print(f"staged {len(glob.glob(f'{IN}/*.nii.gz'))} files")

from nnunetv2.inference.predict_from_raw_data import nnUNetPredictor
pred=nnUNetPredictor(tile_step_size=0.5,use_gaussian=True,use_mirroring=True,device=torch.device('cuda'),verbose=False,allow_tqdm=True)
pred.initialize_from_trained_model_folder(
    f"{base}/nnUNet_results/Dataset503_LiverST2/nnUNetTrainerST2__nnUNetResEncUNetLPlans__3d_fullres",
    use_folds=(0,),checkpoint_name="checkpoint_best.pth")
pred.predict_from_files(IN,OUT,save_probabilities=False,overwrite=False,
                        num_processes_preprocessing=2,num_processes_segmentation_export=2)
for m in glob.glob(f"{OUT}/*.nii.gz"): shutil.copy(m,f"{SAVE}/{os.path.basename(m)}")
print(f"✓ {len(glob.glob(f'{SAVE}/*.nii.gz'))} ST2 masks → {SAVE}")

**Step 17.** Final GED4 crops for all 460 examinations from these masks (`lifs_ged4_hires_train_v2`).

In [ ]:
# === Cell C: re-crop all 460 training cases with uniform ST2 masks ===
import numpy as np, glob, os, SimpleITK as sitk, time

# the two ST2 mask sources, combined
MASK_DIRS = [f"{CKPT}/lifs_train_masks_extra", f"{CKPT}/lifs_train_masks_st2_345"]
RAW="/content/Liver-Train"
OLD_CACHE=f"{CKPT}/lifs_roi_cache"   # for stage/vendor labels
OUT=f"{CKPT}/lifs_ged4_hires_train_v2"  # NEW consistent crops
os.makedirs(OUT,exist_ok=True)

HI=(24,256,256); PAD_FRAC=0.10

# map case -> its ST2 mask
mask_map={}
for md in MASK_DIRS:
    for f in glob.glob(f"{md}/*.nii.gz"):
        mask_map[os.path.basename(f).replace(".nii.gz","")]=f
print(f"ST2 masks available: {len(mask_map)}")

def raw_ged4(case):
    h=glob.glob(f"{RAW}/**/{case}/GED4.nii.gz",recursive=True)
    return h[0] if h else None

cases=sorted(mask_map.keys())
t0=time.time(); done=0
for case in cases:
    out_p=f"{OUT}/{case}.npz"
    if os.path.exists(out_p): done+=1; continue
    gp=raw_ged4(case); mp=mask_map[case]
    if gp is None: print(f"  ✗ no raw: {case}"); continue
    # labels from old cache
    cp=f"{OLD_CACHE}/{case}.npz"
    if not os.path.exists(cp): print(f"  ✗ no label: {case}"); continue
    c=np.load(cp); stage=int(c["stage"]); vendor=str(c["vendor"]); avail=c["avail"]

    img=sitk.ReadImage(gp,sitk.sitkFloat32)
    mask=sitk.ReadImage(mp)
    mask=sitk.Resample(mask,img,sitk.Transform(),sitk.sitkNearestNeighbor,0)
    raw=sitk.GetArrayFromImage(img); mr=sitk.GetArrayFromImage(mask)>0
    if mr.sum()==0: print(f"  ✗ empty: {case}"); continue
    zz,yy,xx=np.where(mr)
    def span(lo,hi,n): p=int((hi-lo)*PAD_FRAC); return max(0,lo-p),min(n,hi+p+1)
    z0,z1=span(zz.min(),zz.max(),raw.shape[0]); y0,y1=span(yy.min(),yy.max(),raw.shape[1]); x0,x1=span(xx.min(),xx.max(),raw.shape[2])
    crop=raw[z0:z1,y0:y1,x0:x1]; ci=sitk.GetImageFromArray(crop)
    ci_r=sitk.Resample(ci,[HI[2],HI[1],HI[0]],sitk.Transform(),sitk.sitkLinear,ci.GetOrigin(),
                       [crop.shape[2]/HI[2],crop.shape[1]/HI[1],crop.shape[0]/HI[0]],ci.GetDirection(),0,sitk.sitkFloat32)
    out=sitk.GetArrayFromImage(ci_r); out=(out-out.mean())/(out.std()+1e-6)
    np.savez_compressed(out_p,ged4=out.astype(np.float32),stage=stage,vendor=vendor,avail=avail)
    done+=1
    if done%50==0: print(f"  {done}/{len(cases)} ({time.time()-t0:.0f}s)")

print(f"\n✓ re-cropped {done}/{len(cases)} with uniform ST2 masks → {OUT}")
from collections import Counter
s=[int(np.load(f)["stage"]) for f in glob.glob(f"{OUT}/*.npz")]
print("stage dist:", dict(sorted(Counter(s).items())), "| total:", len(s))